# UCS420 Cognitive Computing - Assignment 6

## NumPy II

Name: Vishu Verma

Roll number: 1024170242

In [1]:
import numpy as np

## Question 1

Sensor readings, the sensor has an error of +2°C. No for loops used.

In [2]:
temperature = np.array([25, 28, 31, 35, 38, 27, 33, 40])
print("Raw readings:", temperature)

Raw readings: [25 28 31 35 38 27 33 40]


**a) Add 2°C to every reading using vectorization.**

In [3]:
corrected = temperature + 2   # the scalar 2 is broadcast to all 8 readings
print("Corrected readings:", corrected)

Corrected readings: [27 30 33 37 40 29 35 42]


**b) Convert all temperatures from Celsius to Fahrenheit using $F = \frac{9}{5}C + 32$.**

In [4]:
fahrenheit = corrected * 9 / 5 + 32
print("Celsius   :", corrected)
print("Fahrenheit:", fahrenheit)

Celsius   : [27 30 33 37 40 29 35 42]
Fahrenheit: [ 80.6  86.   91.4  98.6 104.   84.2  95.  107.6]


**c) Identify all readings greater than 32°C using Boolean indexing.**

I used the corrected readings since those are the actual temperatures (also printed the raw ones to compare).

In [5]:
mask = corrected > 32
print("Boolean mask      :", mask)
print("Readings above 32 :", corrected[mask])

# the same check on the raw (uncorrected) readings, for comparison
print("\nRaw readings above 32      :", temperature[temperature > 32])

Boolean mask      : [False False  True  True  True False  True  True]
Readings above 32 : [33 37 40 35 42]

Raw readings above 32      : [35 38 33 40]


**d) Count how many readings exceed 32°C.**

In [6]:
print("Readings above 32°C:", np.sum(mask))   # True counts as 1, False as 0
print("Same with count_nonzero:", np.count_nonzero(mask))
print("Raw readings above 32°C (before the +2 correction):", np.sum(temperature > 32))

Readings above 32°C: 5
Same with count_nonzero: 5
Raw readings above 32°C (before the +2 correction): 4


**e) How vectorization and Boolean indexing make processing more efficient**

- Vectorized operations like `temperature + 2` run as one loop in compiled C code inside numpy. A Python for loop handles every value as a separate Python object, which is much slower, especially for big arrays.
- The code is shorter. `corrected[corrected > 32]` does the filtering in one line, no index variable or empty list to append to, so fewer chances of mistakes.
- The boolean mask can be reused to select, count (`mask.sum()`) or change values.
- The same line works for 8 values or millions of values.

I timed both on 1 million random readings below.

In [7]:
import time

big = np.random.default_rng(0).integers(20, 45, size=1_000_000)

t0 = time.perf_counter()
loop_result = [x for x in big if x + 2 > 32]
t1 = time.perf_counter()
vec_result = big[big + 2 > 32]
t2 = time.perf_counter()

print(f"Python loop : {(t1 - t0) * 1000:7.1f} ms")
print(f"Vectorized  : {(t2 - t1) * 1000:7.1f} ms")
print("Same answer :", len(loop_result) == len(vec_result))

Python loop :    57.9 ms
Vectorized  :     4.6 ms
Same answer : True


## Question 2

Steps of 4 users (rows) over 3 days (columns).

In [8]:
steps = np.array([
    [5000, 6200, 7100],
    [8000, 7500, 9000],
    [4500, 5100, 4800],
    [9000, 8500, 9500]])
print(steps)
print("Shape:", steps.shape, "(4 users, 3 days)")

[[5000 6200 7100]
 [8000 7500 9000]
 [4500 5100 4800]
 [9000 8500 9500]]
Shape: (4, 3) (4 users, 3 days)


In [9]:
print("a) Total steps recorded :", steps.sum())
print("b) Mean number of steps :", steps.mean())
print("c) Maximum value        :", steps.max())
print("   Minimum value        :", steps.min())

a) Total steps recorded : 84200
b) Mean number of steps : 7016.666666666667
c) Maximum value        : 9500
   Minimum value        : 4500


In [10]:
# axis=0 collapses the rows, so one total per column (day)
print("d) Total steps per day  :", steps.sum(axis=0))
# axis=1 collapses the columns, so one total per row (user)
print("e) Total steps per user :", steps.sum(axis=1))

d) Total steps per day  : [26500 27300 30400]
e) Total steps per user : [18300 24500 14400 27000]


In [11]:
flat_index = steps.argmax()                            # index in the flattened array
user, day = np.unravel_index(flat_index, steps.shape)  # convert to (row, column)
print("f) argmax() on the flattened array:", flat_index)
print(f"   Position (user, day) = ({user}, {day}) i.e. User {user + 1}, Day {day + 1}, value {steps[user, day]}")

f) argmax() on the flattened array: 11
   Position (user, day) = (3, 2) i.e. User 4, Day 3, value 9500


## Question 3

**a) Create the array.**

In [12]:
original = np.array([1, 2, 3, 4, 5, 6])
print("original:", original)

original: [1 2 3 4 5 6]


**b) Slice from index 1 to 4 and store it in `subset`.**

In [13]:
subset = original[1:5]   # indices 1, 2, 3, 4 (the stop index is excluded)
print("subset:", subset)

subset: [2 3 4 5]


**c) Change the first element of `subset` to 999.**

In [14]:
subset[0] = 999
print("subset  :", subset)
print("original:", original)
print("Shares memory:", np.shares_memory(original, subset))

subset  : [999   3   4   5]
original: [  1 999   3   4   5   6]
Shares memory: True


The original array also changed (index 1 became 999). This is because slicing gives a view, subset uses the same memory as original.

**d) Slice again, this time with `.copy()`, and change its first element to 500.**

In [15]:
copied = original[1:5].copy()
copied[0] = 500
print("copied  :", copied)
print("original:", original)   # unchanged by this step
print("Shares memory:", np.shares_memory(original, copied))

copied  : [500   3   4   5]
original: [  1 999   3   4   5   6]
Shares memory: False


This time original did not change because `.copy()` makes a new array with its own memory.

**e) Numbers 1 to 12 with `np.arange()`, reshaped to 3x4.**

In [16]:
matrix = np.arange(1, 13).reshape(3, 4)
print(matrix)

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]


**f) Indexing and slicing.**

In [17]:
print("First row :", matrix[0])
print("Last row  :", matrix[-1])
print("Second column:", matrix[:, 1])
print("Rows 1-2, columns 2-3:")
print(matrix[1:3, 2:4])

First row : [1 2 3 4]
Last row  : [ 9 10 11 12]
Second column: [ 2  6 10]
Rows 1-2, columns 2-3:
[[ 7  8]
 [11 12]]


Above I took rows 1-2 and columns 2-3 as indices (starting from 0). If it means the 1st-2nd row and 2nd-3rd column instead:

In [18]:
print(matrix[0:2, 1:3])

[[2 3]
 [6 7]]


**g) Flatten with both `flatten()` and `ravel()`.**

In [19]:
flat_f = matrix.flatten()
flat_r = matrix.ravel()
print("flatten():", flat_f)
print("ravel()  :", flat_r)

flatten(): [ 1  2  3  4  5  6  7  8  9 10 11 12]
ravel()  : [ 1  2  3  4  5  6  7  8  9 10 11 12]


**h) Modify the array returned by `ravel()`.**

In [20]:
flat_r[0] = 100
print("ravel() array:", flat_r)
print("matrix after changing the ravel() array:")
print(matrix)

ravel() array: [100   2   3   4   5   6   7   8   9  10  11  12]
matrix after changing the ravel() array:
[[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]


Yes, the original matrix changed (matrix[0,0] is now 100) because ravel() returns a view when it can.

**i) Modify the array returned by `flatten()`.**

In [21]:
flat_f[1] = 200
print("flatten() array:", flat_f)
print("matrix after changing the flatten() array:")
print(matrix)

flatten() array: [  1 200   3   4   5   6   7   8   9  10  11  12]
matrix after changing the flatten() array:
[[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]


No change in the matrix this time (matrix[0,1] is still 2) because flatten() always returns a copy.

**j) shape, ndim, size and dtype of the matrix**

In [22]:
print("shape:", matrix.shape)
print("ndim :", matrix.ndim)
print("size :", matrix.size)
print("dtype:", matrix.dtype)

shape: (3, 4)
ndim : 2
size : 12
dtype: int64


## Question 4

Predicting assistance score from sleep hours, activity level and stress level. Each row is a user and each column is a feature.

In [23]:
X = np.array([
    [6, 70, 3],
    [5, 50, 6],
    [8, 80, 2],
    [4, 30, 8]
])
y = np.array([40, 65, 30, 85])

**a) Shape and dimensions of X.**

In [24]:
print("Shape:", X.shape, "(4 users, 3 features)")
print("ndim :", X.ndim)

Shape: (4, 3) (4 users, 3 features)
ndim : 2


**b) Transpose of X.**

In [25]:
print(X.T)
print('Shape:', X.T.shape)

[[ 6  5  8  4]
 [70 50 80 30]
 [ 3  6  2  8]]
Shape: (3, 4)


Transpose swaps the rows and columns. In X each row was a user, in X.T each row is a feature - first row has sleep hours of all users, second row activity, third row stress.

**c) Matrix product `X.T @ X`.**

In [26]:
XtX = X.T @ X
print(XtX)
print("Shape:", XtX.shape)

[[  141  1430    96]
 [ 1430 14700   910]
 [   96   910   113]]
Shape: (3, 3)


The result is a 3x3 symmetric matrix.

**d) Matrix inverse with `np.linalg.inv()`.**

In [27]:
# inverse exists only if the matrix is square and determinant is not 0
print("Determinant of X.T @ X:", np.linalg.det(XtX))
print("Rank:", np.linalg.matrix_rank(XtX), "of", XtX.shape[0])

XtX_inv = np.linalg.inv(XtX)
print("\nInverse:")
print(XtX_inv)
print("\nCheck, (X.T @ X) @ inverse should be identity:")
print(np.round(XtX @ XtX_inv, 6) + 0.0)   # +0.0 to remove -0.

Determinant of X.T @ X: 753700.0000000029
Rank: 3 of 3

Inverse:
[[ 1.10521428 -0.09848746 -0.14581398]
 [-0.09848746  0.00891203  0.01190129]
 [-0.14581398  0.01190129  0.0368847 ]]

Check, (X.T @ X) @ inverse should be identity:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


X is 4x3 so it can't be inverted directly, but X.T @ X is 3x3 with non-zero determinant so its inverse exists.

**e) Ordinary Least Squares: $\beta = (X^TX)^{-1}X^Ty$.**

In [28]:
beta = XtX_inv @ X.T @ y
print("beta =", beta)

# checking with numpy's lstsq
beta_check, *_ = np.linalg.lstsq(X, y, rcond=None)
print("lstsq gives:", beta_check)
print("Match:", np.allclose(beta, beta_check))

beta = [-0.15921454  0.14083853 10.06302242]
lstsq gives: [-0.15921454  0.14083853 10.06302242]
Match: True


In [29]:
y_hat = X @ beta
print("Actual    :", y)
print("Predicted :", np.round(y_hat, 2))
print("Residuals :", np.round(y - y_hat, 2))

Actual    : [40 65 30 85]
Predicted : [39.09 66.62 30.12 84.09]
Residuals : [ 0.91 -1.62 -0.12  0.91]


**f) What the coefficients mean.**

In [30]:
for name, b in zip(["Sleep hours", "Activity level", "Stress level"], beta):
    print(f"{name:15s}: {b:+.4f}")

Sleep hours    : -0.1592
Activity level : +0.1408
Stress level   : +10.0630


score = b1*sleep + b2*activity + b3*stress

Each coefficient tells how much the score changes when that feature increases by 1 and the other two stay the same.

- Sleep (-0.16): one more hour of sleep reduces the score by about 0.16, very small effect.
- Activity (+0.14): one more unit of activity increases the score by 0.14. This looks odd because active users have lower scores in the data, but activity and stress are closely related (active users are less stressed), so the stress coefficient already covers most of it.
- Stress (+10.06): one more point of stress increases the score by about 10. This is the biggest factor by far.

There is no intercept term, and there are only 4 users for 3 coefficients, so these values only really fit this small dataset.

**g) Predict the score for a new user.**

In [31]:
new_user = np.array([5, 40, 7])
prediction = new_user @ beta
print("New user [sleep, activity, stress]:", new_user)
print(f"Predicted assistance score: {prediction:.2f}")

New user [sleep, activity, stress]: [ 5 40  7]
Predicted assistance score: 75.28


Predicted score is about 75.3. Stress is 7 which is between user 2 (stress 6, score 65) and user 4 (stress 8, score 85), so the result makes sense.